In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input

from tensorflow.keras.callbacks import Callback, EarlyStopping

## Simulation1

In [ ]:
X_id = pd.read_excel('Target1.xlsx', index_col=0, sheet_name="identity").values.astype(np.float32)
X_pos = pd.read_excel('Target1.xlsx', index_col=0, sheet_name="position").values.astype(np.float32)
X_allog = pd.read_excel('Targetx.xlsx', index_col=0,sheet_name="allog").values.astype(np.float32)
Y = pd.read_excel('orthograph_sim1_2.xlsx').values[0:477,0:350].astype(np.float32)


in_id   = Input(shape=(35,),  name='in_id')
tower_id  = Dense(700, activation='sigmoid')(in_id)


in_pos  = Input(shape=(100,), name='in_pos')
tower_pos = Dense(700, activation="sigmoid")(in_pos)

in_allog  = Input(shape=(116,), name='in_allog')
tower_allog = Dense(700, activation="sigmoid")(in_allog)

combined = concatenate([tower_id, tower_pos, tower_allog])
h = Dense(500, activation="sigmoid")(combined)
output = Dense(350, activation="sigmoid")(h)

model1 = Model(inputs=[in_id, in_pos, in_allog], outputs=output)


model1.compile(optimizer="adam", loss="mse")
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)

model1.fit([X_id, X_pos,X_allog],Y,batch_size=25, epochs=700, callbacks=[callback])
model1.save('simulation_1a_model.h5')

Epoch 1/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.1041
Epoch 2/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0178
Epoch 3/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0180
Epoch 4/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0177
Epoch 5/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0176
Epoch 6/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0178
Epoch 7/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.0177
Epoch 8/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0177
Epoch 9/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0175
Epoch 10/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0177
Epoch 11/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0177
Epoch 12/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0176
Epoch 13/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0173
Epoch 14/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0177
Epoch 15/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - lo

# Test the model
We will find the correlation distance between the original word and its different modified forms.

In [ ]:
data_lst = ["TL+Allog1.xlsx", "TL-Allog1.xlsx", "SubL+Allog1.xlsx", "SubL-Allog1.xlsx"]

# orthographic vectors
Y = pd.read_excel('orthograph_sim1_2.xlsx').values[0:477,0:350].astype(np.float32)

# output DataFrame and a dict to hold preds
df_sim1 = pd.DataFrame()
preds = {}


for doc in data_lst:

    key = doc.replace('1.xlsx','').lower()
    X_id    = pd.read_excel(doc, sheet_name='identity', index_col=0).values.astype(np.float32)
    X_pos   = pd.read_excel(doc, sheet_name='position', index_col=0).values.astype(np.float32)
    allog_f = doc.replace('1.xlsx','x.xlsx')
    X_allog   = pd.read_excel(allog_f, sheet_name='allog', index_col=0).values.astype(np.float32)
    pred = model1.predict([X_id, X_pos, X_allog])
    preds[key] = pred

    # per-word correlation distances
    valz = []
    for i in range(len(Y)):
        r = np.corrcoef(pred[i], Y[i])[0,1]
        valz.append(round(1-r,3))

    df_sim1[f'target-{key}'] = valz


pairs = [('tl-allog',  'subl-allog'),('tl+allog', 'subl+allog'),]
for a, b in pairs:
    df_sim1[f'{a}-{b}'] = [
        round(1 - np.corrcoef(preds[a][i], preds[b][i])[0,1], 4)
        for i in range(len(Y))]


df_sim1.to_excel('simulation1_correlation_test_10.xlsx')


15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


# Correlation Distance Between words that share the same root

In [ ]:
import itertools

df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model1.predict([X_id, X_pos, X_allog])

df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


# Load roots file
df_roots = pd.read_excel('targets with roots and semantics.xlsx', sheet_name='grouped by roots',     index_col=0)


# Merge so each row has its vector
df_merged = df_roots.merge(df_pred, on='word', how='inner')


# compute all pairwise distances for each root
rows = []
for root, grp in df_merged.groupby('Root'):
    items = list(zip(grp['word'], grp[[f'v{i}' for i in range(350)]].values))
    for (w1, vec1), (w2, vec2) in itertools.combinations(items, 2):
        r = np.corrcoef(vec1, vec2)[0,1]
        dist = 1 - r
        rows.append({'Root': root, 'word1': w1, 'word2': w2, 'distance': dist})

df_pairs = pd.DataFrame(rows)

# Optionally - summary stats per root
summary = df_pairs.groupby('Root')['distance'].agg(
    mean_dist='mean',
    std_dist ='std',
    min_dist ='min',
    max_dist ='max'
).reset_index()


with pd.ExcelWriter('simulation1_root_pairwise_distances.xlsx') as wr:
    df_pairs.to_excel(wr, sheet_name='all_pairs', index=False)
    summary.to_excel(wr, sheet_name='per_root_summary', index=False)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


# Correlation Distance Between the word and other 4 conditions

In [ ]:
import numpy as np
import pandas as pd

df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model1.predict([X_id, X_pos, X_allog])


df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words



PAIRS_XLSX  = 'targets with roots and semantics.xlsx'
PAIRS_SHEET = '4_conditions'


vec_cols = [c for c in df_pred.columns if c.startswith('v')]
vec_map  = dict(zip(df_pred['word'], df_pred[vec_cols].values))

# read pairs
pairs = pd.read_excel(PAIRS_XLSX, sheet_name=PAIRS_SHEET)

def corr_distance(w1, w2):
    v1 = vec_map.get(w1)
    v2 = vec_map.get(w2)
    if v1 is None or v2 is None:
        return np.nan
    r = np.corrcoef(v1, v2)[0, 1]
    return 1 - r

pairs['distance'] = pairs.apply(lambda r: corr_distance(r['word1'], r['word2']), axis=1)

# keep exactly 4 columns in this order
out = pairs[['word1', 'word2', 'condition', 'distance']]
out.to_excel('simulation1_4_conditions_pairs_with_distance_10.xlsx', index=False)



15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
